# M02-P01 — Enterprise Ticket Classifier

En M01 utilizamos un LLM. En esta práctica hacemos algo deliberadamente distinto: construiremos un clasificador **clásico**, pequeño e interpretable con **TF-IDF + Logistic Regression**.

Queremos abrir la caja: ver los datos, cómo el texto se convierte en números, cómo se separan entrenamiento y test, qué errores comete el modelo y con qué probabilidad clasifica.

> Esta práctica no usa AWS ni Bedrock. Todo se ejecuta localmente en el Codespace.


## 1. Preparar el laboratorio

Usaremos un dataset ficticio de soporte con cinco colas: acceso, red, aplicaciones, seguridad y hardware.

El dataset es pequeño a propósito. Sirve para entender el proceso; no representa un clasificador listo para producción.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)

from enterprise_genai.classical_ml import (
    LABEL_DISPLAY,
    LABEL_ORDER,
    active_tfidf_terms,
    build_ticket_classifier,
    load_ticket_dataset,
    predict_with_probabilities,
    split_ticket_dataset,
    top_terms_by_class,
)

pd.set_option("display.max_colwidth", 120)


## 2. Mirar los datos antes de entrenar

En aprendizaje supervisado cada ejemplo tiene:

```text
texto del ticket + etiqueta correcta
```

La etiqueta es lo que el modelo intentará aprender a predecir.


In [ ]:
df = load_ticket_dataset()

print("Tickets:", len(df))
print("Categorías:", sorted(df["category"].unique()))
display(df.head(8))

class_counts = (
    df["category"]
    .value_counts()
    .reindex(LABEL_ORDER)
    .rename_axis("category")
    .reset_index(name="tickets")
)
class_counts["label"] = class_counts["category"].map(LABEL_DISPLAY)
class_counts


### Qué debes observar

Tenemos **80 tickets**, 16 por categoría. Está balanceado deliberadamente para que en esta práctica las métricas no queden dominadas por una clase mucho más frecuente.

En datos empresariales reales esto no suele ser tan limpio.


## 3. Separar entrenamiento y test

No evaluamos un modelo con los mismos ejemplos que ha usado para aprender.

Hacemos una división:

```text
75 % entrenamiento
25 % test
```

Usamos `stratify` para conservar la proporción de clases y un `random_state` fijo para que todos obtengamos la misma partición.


In [ ]:
X_train, X_test, y_train, y_test = split_ticket_dataset(df)

split_summary = pd.DataFrame(
    {
        "train": y_train.value_counts().reindex(LABEL_ORDER),
        "test": y_test.value_counts().reindex(LABEL_ORDER),
    }
)
split_summary.index = [LABEL_DISPLAY[label] for label in split_summary.index]

print("Train:", len(X_train))
print("Test:", len(X_test))
split_summary


## 4. TF-IDF: convertir texto en números

Logistic Regression no recibe frases directamente. Necesita números.

TF-IDF crea un vocabulario a partir de los tickets de entrenamiento y representa cada ticket mediante pesos numéricos para palabras y pares de palabras.

En esta práctica usamos:

```text
unigramas  → "vpn"
bigramas   → "código mfa"
```

La matriz resultante suele ser **dispersa**: cada ticket solo activa una pequeña parte del vocabulario total.


In [ ]:
model = build_ticket_classifier()
model.fit(X_train, y_train)

vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

X_train_tfidf = vectorizer.transform(X_train)
vocabulary_size = len(vectorizer.get_feature_names_out())
total_cells = X_train_tfidf.shape[0] * X_train_tfidf.shape[1]
sparsity = 1 - (X_train_tfidf.nnz / total_cells)

print("Matriz TF-IDF:", X_train_tfidf.shape)
print("Vocabulario:", vocabulary_size, "términos")
print("Valores no cero:", X_train_tfidf.nnz)
print(f"Dispersión (ceros): {sparsity:.1%}")


### Abrimos un ticket concreto

Veamos qué términos del vocabulario quedan activos y con qué peso TF-IDF.


In [ ]:
sample_text = X_train.iloc[0]
print("Ticket:")
print(sample_text)

active_tfidf_terms(model, sample_text, n=12)


## 5. Logistic Regression aprende pesos por categoría

TF-IDF representa el texto. Logistic Regression aprende qué combinaciones de esos términos favorecen cada clase.

Después puede producir:

```text
clase predicha
+
probabilidad para cada clase
```

Ahora evaluamos **solo con el conjunto de test**, que no se usó para ajustar el modelo.


In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.1%}")

report = classification_report(
    y_test,
    y_pred,
    labels=LABEL_ORDER,
    target_names=[LABEL_DISPLAY[label] for label in LABEL_ORDER],
    output_dict=True,
    zero_division=0,
)

report_df = (
    pd.DataFrame(report)
    .transpose()
    .round(3)
)

report_df


### Precision, recall y F1

- **Precision**: cuando el modelo predice una clase, ¿con qué frecuencia acierta?
- **Recall**: de los casos que realmente pertenecen a una clase, ¿cuántos encuentra?
- **F1**: combina precision y recall en una sola medida.

No buscamos una métrica perfecta. Con un dataset tan pequeño queremos que existan errores para poder estudiarlos.


## 6. Matriz de confusión

La diagonal representa aciertos. Las celdas fuera de la diagonal muestran qué categorías se confunden.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=LABEL_ORDER,
    display_labels=[LABEL_DISPLAY[label] for label in LABEL_ORDER],
    xticks_rotation=35,
    ax=ax,
)
ax.set_title("Matriz de confusión — conjunto de test")
plt.tight_layout()
plt.show()


## 7. Los errores son más interesantes que el accuracy

Inspeccionamos los tickets que el modelo clasificó mal.

Añadimos la probabilidad máxima. Recuerda: una probabilidad alta es **confianza del modelo**, no una garantía de que tenga razón.


In [ ]:
test_probabilities = model.predict_proba(X_test)
max_probability = test_probabilities.max(axis=1)

errors = pd.DataFrame(
    {
        "ticket": X_test.values,
        "real": y_test.values,
        "predicha": y_pred,
        "prob_max": max_probability,
    }
)

errors = errors[errors["real"] != errors["predicha"]].copy()
errors["real"] = errors["real"].map(LABEL_DISPLAY)
errors["predicha"] = errors["predicha"].map(LABEL_DISPLAY)
errors["prob_max"] = errors["prob_max"].map(lambda value: f"{value:.1%}")

print("Errores en test:", len(errors))
errors


### Preguntas para interpretar los errores

No basta con decir “se ha equivocado”.

Busca pistas:

- ¿aparecen palabras asociadas a otra categoría?
- ¿es realmente un ticket ambiguo?
- ¿faltan ejemplos similares en entrenamiento?
- ¿el modelo está usando correlaciones superficiales?

Este análisis será importante en M02-P02 cuando comparemos con un LLM.


## 8. ¿Qué términos ha aprendido el modelo?

Como Logistic Regression es relativamente interpretable, podemos mirar los términos con mayor peso positivo para cada clase.

No significa que una palabra determine por sí sola la categoría. Son asociaciones aprendidas dentro de este dataset.


In [ ]:
top_terms = top_terms_by_class(model, n=8)

for category in LABEL_ORDER:
    print("\n", LABEL_DISPLAY[category])
    display(
        top_terms[top_terms["category"] == category][["rank", "term", "weight"]]
        .style.format({"weight": "{:.3f}"})
    )


## 9. Probar tickets nuevos

Ahora usamos el modelo como lo haría una aplicación.

Para cada ticket mostramos las dos clases con mayor probabilidad.


In [ ]:
new_tickets = [
    "No puedo acceder al CRM porque el código MFA no llega al móvil.",
    "La VPN conecta pero hay pérdida de paquetes y mucha latencia.",
    "El EDR detectó un script sospechoso de PowerShell en el portátil.",
    "El monitor del dock parpadea y deja de mostrar imagen.",
    "El ERP devuelve un error al aprobar una orden de compra.",
]

rows = []

for text in new_tickets:
    probs = predict_with_probabilities(model, text)
    rows.append(
        {
            "ticket": text,
            "predicción": probs.iloc[0]["label"],
            "prob_1": probs.iloc[0]["probability"],
            "segunda": probs.iloc[1]["label"],
            "prob_2": probs.iloc[1]["probability"],
        }
    )

pd.DataFrame(rows).style.format(
    {
        "prob_1": "{:.1%}",
        "prob_2": "{:.1%}",
    }
)


## 10. Casos ambiguos y fuera de distribución

Un clasificador cerrado **siempre tiene que elegir una de sus cinco clases**, aunque el ticket no encaje bien.

Probamos dos casos deliberadamente incómodos:

1. mezcla problemas de red y hardware;
2. habla de reservar una sala, algo que no pertenece a ninguna clase del entrenamiento.

Mira especialmente la probabilidad máxima y la distancia respecto a la segunda clase.


In [ ]:
challenge_tickets = [
    "Después de conectar la VPN el portátil pierde el Wi-Fi y el adaptador USB deja de responder.",
    "Necesito reservar una sala grande para una reunión con clientes el próximo martes.",
]

challenge_rows = []

for text in challenge_tickets:
    probs = predict_with_probabilities(model, text)
    challenge_rows.append(
        {
            "ticket": text,
            "predicción": probs.iloc[0]["label"],
            "prob_1": probs.iloc[0]["probability"],
            "segunda": probs.iloc[1]["label"],
            "prob_2": probs.iloc[1]["probability"],
            "margen": probs.iloc[0]["probability"] - probs.iloc[1]["probability"],
        }
    )

pd.DataFrame(challenge_rows).style.format(
    {
        "prob_1": "{:.1%}",
        "prob_2": "{:.1%}",
        "margen": "{:.1%}",
    }
)


### Importante: probabilidad no es conocimiento del mundo

Si escribimos un ticket totalmente ajeno al entrenamiento, Logistic Regression no puede responder:

```text
"no conozco esta categoría"
```

porque su espacio de salida está cerrado.

Podemos añadir reglas como un umbral de confianza, pero el umbral es una decisión de diseño y debe evaluarse con datos.

En M02-P02 compararemos este comportamiento con un LLM.


## 11. Reentrenar después de evaluar

Para evaluar correctamente hemos reservado test.

Una vez tomada la decisión de usar el enfoque, es habitual entrenar una versión final con todos los ejemplos etiquetados disponibles.

La interfaz Streamlit de esta práctica hace justamente eso: entrena con los 80 tickets y nos deja probar textos interactivamente.


In [ ]:
print("Ejecuta en un terminal:")
print("streamlit run labs/M02/P01/ticket_classifier_app.py")


## 12. Mini-reto

En la interfaz prueba al menos tres tickets:

1. uno muy claro;
2. uno ambiguo entre dos categorías;
3. uno que no pertenezca realmente a ninguna categoría.

Para cada uno, observa:

- predicción;
- probabilidad máxima;
- distribución completa de probabilidades;
- términos TF-IDF activos;
- términos con mayor peso de la clase predicha.

No modifiques el modelo todavía. Primero intenta explicar **por qué** se comporta así.


## 13. Qué debemos conservar

El pipeline completo es:

```text
tickets etiquetados
      ↓
train / test
      ↓
TF-IDF
      ↓
vectores numéricos
      ↓
Logistic Regression
      ↓
clase + probabilidades
      ↓
métricas + análisis de errores
```

Este enfoque tiene ventajas claras:

- rápido;
- barato;
- local;
- reproducible;
- medible;
- relativamente interpretable.

Y también limitaciones:

- necesita ejemplos etiquetados;
- depende mucho del dominio y vocabulario de entrenamiento;
- solo conoce las clases definidas;
- puede fallar con lenguaje ambiguo o fuera de distribución.

En M02-P02 usaremos exactamente estas limitaciones y fortalezas para compararlo con un LLM, sin asumir que uno de los dos enfoques es siempre mejor.
